### Feature Engineering

In [1]:
import pandas as pd

df = pd.read_csv("clean_data_after_eda.csv")
df.head()

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,9.084737e-04,2.086294,99.530517,44.235794,2.086425,9.953056e+01,4.423670e+01,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000e+00,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000e+00,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000e+00,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000e+00,0
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,0.000003,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000003,0.000000e+00,0.000000e+00,0
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,0.000011,2.896760e-06,4.860000e-10,0.000000,0.000000,0.000000,0.000011,2.896760e-06,4.860000e-10,0


In [2]:
df.columns

Index(['id', 'channel_sales', 'cons_12m', 'cons_gas_12m', 'cons_last_month',
       'date_activ', 'date_end', 'date_modif_prod', 'date_renewal',
       'forecast_cons_12m', 'forecast_cons_year', 'forecast_discount_energy',
       'forecast_meter_rent_12m', 'forecast_price_energy_off_peak',
       'forecast_price_energy_peak', 'forecast_price_pow_off_peak', 'has_gas',
       'imp_cons', 'margin_gross_pow_ele', 'margin_net_pow_ele', 'nb_prod_act',
       'net_margin', 'num_years_antig', 'origin_up', 'pow_max',
       'var_year_price_off_peak_var', 'var_year_price_peak_var',
       'var_year_price_mid_peak_var', 'var_year_price_off_peak_fix',
       'var_year_price_peak_fix', 'var_year_price_mid_peak_fix',
       'var_year_price_off_peak', 'var_year_price_peak',
       'var_year_price_mid_peak', 'var_6m_price_off_peak_var',
       'var_6m_price_peak_var', 'var_6m_price_mid_peak_var',
       'var_6m_price_off_peak_fix', 'var_6m_price_peak_fix',
       'var_6m_price_mid_peak_fix', 'var_6m_p

### price sensitivity

In [3]:
df["price_diff_offpeak_peak"] = (
    df["forecast_price_energy_off_peak"]
    - df["forecast_price_energy_peak"]
)

### Consumption per Active Product

In [4]:
df["cons_per_product"] = (
    df["cons_12m"] / (df["nb_prod_act"] + 1)
)

### Margin per Consumption

In [5]:
df["margin_per_cons"] = (
    df["net_margin"] / (df["cons_12m"] + 1)
)

In [6]:
df["avg_monthly_cons"] = (
    df["cons_12m"] / 12
)

In [9]:
date_cols = [
    "date_activ",
    "date_end",
    "date_modif_prod",
    "date_renewal"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

reference_date = df["date_activ"].max()

df["tenure_years"] = (
    reference_date - df["date_activ"]
).dt.days / 365.25

In [10]:
new_features = [
    "price_diff_offpeak_peak",
    "cons_per_product",
    "margin_per_cons",
    "avg_monthly_cons",
    "tenure_years"
]

df[new_features].head()

,price_diff_offpeak_peak,cons_per_product,margin_per_cons,avg_monthly_cons,tenure_years
0,0.016339,0.0,678.990000,0.000000,1.212868
1,0.145711,2330.0,0.004053,388.333333,5.029432
2,0.077895,272.0,0.012110,45.333333,4.377823
3,0.146694,792.0,0.016063,132.000000,4.424367
4,0.016885,2212.5,0.010840,368.750000,4.632444


In [11]:
corr = df.corr(numeric_only=True)["churn"]

print(corr.sort_values(ascending=False))

churn                             1.000000
margin_net_pow_ele                0.095772
margin_gross_pow_ele              0.095725
forecast_meter_rent_12m           0.044245
net_margin                        0.041135
pow_max                           0.030362
forecast_price_energy_peak        0.029315
var_year_price_off_peak_var       0.028646
margin_per_cons                   0.023771
var_6m_price_off_peak_var         0.019628
var_year_price_off_peak           0.018930
var_year_price_off_peak_fix       0.018930
forecast_discount_energy          0.017026
forecast_price_pow_off_peak       0.014778
var_year_price_peak_fix           0.014674
var_year_price_peak               0.014674
var_6m_price_peak                 0.013521
var_6m_price_peak_fix             0.013521
forecast_cons_12m                 0.012949
var_6m_price_off_peak             0.011891
var_6m_price_off_peak_fix         0.011891
var_year_price_mid_peak_var       0.010415
var_6m_price_mid_peak_var         0.010223
var_6m_pric

## Feature engineering was performed to create variables that may better capture customer behavior and price sensitivity. New features include customer tenure, average monthly consumption, consumption per active product, margin per consumption, and the difference between off-peak and peak energy prices. These features are expected to improve the predictive power of the churn model by incorporating customer loyalty, profitability, usage patterns, and sensitivity to pricing changes.